# Clustering Crédit - Partie 1: Exploration & Préparation Interactive

🎯 **Objectif**: Comprendre les données clients et créer des variables métier pour la segmentation

📋 **Ce que vous allez faire**:
1. Charger et explorer le dataset de crédit
2. Analyser les distributions et relations entre variables
3. Créer des indicateurs métier (utilisation, remboursement, risque)
4. Préparer les données pour le clustering

💡 **Cas d'usage**: Vous êtes Data Scientist dans une banque. Le Directeur Marketing veut segmenter les clients pour personnaliser les offres.

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, SelectMultiple, interactive
import os
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("🚀 Environnement prêt pour l'exploration!")

🚀 Environnement prêt pour l'exploration!


## 📊 Étape 1: Chargement des données

Le dataset contient des informations sur des clients de cartes de crédit à Taïwan.

In [8]:
def load_credit_data(local_path=None, sample_n=1000):
    """Charge le dataset avec fallback robuste"""
    def _read_any(path):
        ext = os.path.splitext(path)[1].lower()
        if ext in ['.csv', '.txt']:
            return pd.read_csv(path)
        else:
            try:
                return pd.read_excel(path, header=1, index_col=0)
            except:
                return pd.read_excel(path)

    if local_path and os.path.exists(local_path):
        df = _read_any(local_path)
        print(f"✅ Données chargées depuis: {local_path}")
    else:
        try:
            uci_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls"
            df = pd.read_excel(uci_url, header=1, index_col=0)
            print("✅ Données chargées depuis UCI")
        except:
            raise Exception("❌ Échec du chargement. Spécifiez un chemin local.")
    
    if sample_n and len(df) > sample_n:
        df = df.sample(sample_n, random_state=42)
        print(f"📝 Échantillon de {sample_n} lignes sélectionné")
    
    return df

# Chargement
try:
    df = load_credit_data(sample_n=1000)
    print(f"📏 Dimensions: {df.shape}")
    display(df.head())
except Exception as e:
    print(f"⚠️ Erreur: {e}")
    print("💡 Conseil: téléchargez le fichier UCI et spécifiez le chemin local")

✅ Données chargées depuis UCI
📝 Échantillon de 1000 lignes sélectionné
📏 Dimensions: (1000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
ID,,,,,,,,,,,,,,,,,,,,,
2309,30000,1,2,2,25,0,0,0,0,0,...,12580,13716,14828,1500,2000,1500,1500,1500,2000,0
22405,150000,2,1,2,26,0,0,0,0,0,...,101581,77741,77264,4486,4235,3161,2647,2669,2669,0
23398,70000,2,3,1,32,0,0,0,0,0,...,69753,70111,70212,2431,3112,3000,2438,2500,2554,0
25059,130000,1,3,2,49,0,0,0,0,0,...,16898,11236,6944,1610,1808,7014,27,7011,4408,0
2665,50000,2,2,2,36,0,0,0,0,0,...,19574,20295,19439,2000,1500,1000,1800,0,1000,1


## 📖 Dictionnaire des Données

### Informations générales:
- **Source**: UCI Machine Learning Repository
- **Pays**: Taïwan  
- **Période**: Avril 2005
- **Taille originale**: ~30,000 clients
- **Objectif original**: Prédiction de défaut de paiement (non utilisé ici)

### Variables socio-démographiques:
| Variable | Type | Description | Valeurs |
|----------|------|-------------|---------|
| **AGE** | Numérique | Âge du client | En années |
| **SEX** | Catégorie | Sexe | 1=Homme, 2=Femme |
| **EDUCATION** | Catégorie | Niveau d'éducation | 1=Études supérieures, 2=Université, 3=Lycée, 4=Autres |
| **MARRIAGE** | Catégorie | Statut marital | 1=Marié, 2=Célibataire, 3=Autres |

### Variables financières:
| Variable | Type | Description | Unité |
|----------|------|-------------|-------|
| **LIMIT_BAL** | Numérique | Limite de crédit accordée | Dollars taïwanais (NT$) |
| **BILL_AMT1-6** | Numérique | Montants facturés (6 derniers mois) | NT$ (1=plus récent) |
| **PAY_AMT1-6** | Numérique | Montants remboursés (6 derniers mois) | NT$ (1=plus récent) |

### Variables de comportement de paiement:
| Variable | Type | Description | Valeurs |
|----------|------|-------------|---------|
| **PAY_0, PAY_2-6** | Catégorie | Statut de remboursement | -1=Payé intégralement, 1=Retard 1 mois, 2=Retard 2 mois, etc. |

### Variables métier créées (Feature Engineering):
| Variable | Type | Description | Calcul |
|----------|------|-------------|--------|
| **utilization** | Numérique | Taux d'utilisation du crédit | BILL_AMT1 / LIMIT_BAL |
| **repay_ratio_recent** | Numérique | Ratio de remboursement récent | PAY_AMT1 / BILL_AMT1 |
| **repay_ratio_mean** | Numérique | Ratio de remboursement moyen | Σ(PAY_AMT) / Σ(BILL_AMT) |
| **nb_retards** | Numérique | Nombre de mois en retard | Σ(PAY_x > 0) |
| **retard_recent** | Binaire | Retard dans le mois actuel | PAY_0 > 0 |
| **facture_moyenne** | Numérique | Montant moyen facturé | Moyenne(BILL_AMT1-6) |
| **paiement_moyen** | Numérique | Montant moyen remboursé | Moyenne(PAY_AMT1-6) |
| **tendance_facture** | Numérique | Évolution des factures | (BILL_AMT1 - BILL_AMT3) / BILL_AMT3 |
| **tendance_paiement** | Numérique | Évolution des paiements | (PAY_AMT1 - PAY_AMT3) / PAY_AMT3 |

### 💡 Points d'attention:
- **Échelles différentes**: Les montants sont en NT$, standardisation nécessaire
- **Valeurs manquantes**: Rares dans ce dataset
- **Outliers**: Possibles sur les montants et limites
- **Corrélations**: BILL_AMT et PAY_AMT sont naturellement corrélés

In [3]:
# Aperçu du dictionnaire de données avec statistiques
print("📊 Aperçu statistique du dataset:")
print(f"📏 Dimensions: {df.shape[0]} lignes × {df.shape[1]} colonnes")

# Types de variables
print(f"\n🔢 Types de variables:")
type_counts = df.dtypes.value_counts()
for dtype, count in type_counts.items():
    print(f"  • {dtype}: {count} variables")

# Valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
if missing.sum() > 0:
    print(f"\n❌ Valeurs manquantes:")
    for col, pct in missing_pct[missing_pct > 0].items():
        print(f"  • {col}: {missing[col]} ({pct}%)")
else:
    print(f"\n✅ Aucune valeur manquante détectée")

# Aperçu des variables principales
print(f"\n📈 Statistiques des variables clés:")
key_vars = ['AGE', 'LIMIT_BAL', 'BILL_AMT1', 'PAY_AMT1']
available_vars = [v for v in key_vars if v in df.columns]
if available_vars:
    display(df[available_vars].describe().round(0))

# Distribution des variables catégorielles
cat_vars = ['SEX', 'EDUCATION', 'MARRIAGE']
available_cat = [v for v in cat_vars if v in df.columns]
if available_cat:
    print(f"\n📊 Distribution des variables catégorielles:")
    for var in available_cat:
        print(f"\n{var}:")
        print(df[var].value_counts().head())

📊 Aperçu statistique du dataset:
📏 Dimensions: 1000 lignes × 24 colonnes

🔢 Types de variables:
  • int64: 24 variables

✅ Aucune valeur manquante détectée

📈 Statistiques des variables clés:


,AGE,LIMIT_BAL,BILL_AMT1,PAY_AMT1
count,1000.0,1000.0,1000.0,1000.0
mean,35.0,163280.0,48647.0,5157.0
std,9.0,128382.0,70217.0,12160.0
min,21.0,10000.0,-7438.0,0.0
25%,28.0,50000.0,3895.0,1000.0
50%,34.0,140000.0,19728.0,2100.0
75%,41.0,230000.0,67238.0,5000.0
max,70.0,710000.0,564757.0,152000.0



📊 Distribution des variables catégorielles:

SEX:
SEX
2    598
1    402
Name: count, dtype: int64

EDUCATION:
EDUCATION
2    486
1    344
3    157
5      6
4      5
Name: count, dtype: int64

MARRIAGE:
MARRIAGE
2    533
1    456
3      9
0      2
Name: count, dtype: int64


## 🔍 Étape 2: Exploration Interactive

Utilisez les widgets ci-dessous pour explorer les données.

In [ ]:
@interact(
    colonne=Dropdown(options=list(df.columns), value='LIMIT_BAL', description='Variable:'),
    graphique=Dropdown(options=['Histogramme', 'Boxplot', 'Statistiques'], value='Histogramme', description='Vue:')
)
def explorer_variable(colonne, graphique):
    """Widget interactif pour explorer une variable"""
    plt.figure(figsize=(10, 4))
    
    if graphique == 'Histogramme':
        plt.subplot(1, 2, 1)
        df[colonne].hist(bins=30, alpha=0.7, edgecolor='black')
        plt.title(f'Distribution de {colonne}')
        plt.xlabel(colonne)
        plt.ylabel('Fréquence')
        
        plt.subplot(1, 2, 2)
        df[colonne].plot(kind='box')
        plt.title(f'Boxplot de {colonne}')
        
    elif graphique == 'Boxplot':
        df[colonne].plot(kind='box', vert=False)
        plt.title(f'Boxplot de {colonne}')
        
    elif graphique == 'Statistiques':
        stats = df[colonne].describe()
        print("📊 Statistiques descriptives:")
        for stat, val in stats.items():
            print(f"  {stat}: {val:.2f}")
        
        print(f"\n🔢 Valeurs manquantes: {df[colonne].isna().sum()}")
        print(f"🔢 Valeurs uniques: {df[colonne].nunique()}")
        
        if df[colonne].nunique() < 20:
            print(f"\n📋 Répartition des valeurs:")
            print(df[colonne].value_counts().head(10))
    
    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='Variable:', options=('LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', …

## 🛠️ Étape 3: Feature Engineering Interactif

Créons des indicateurs métier pour mieux comprendre le comportement client.

In [10]:
def creer_features_metier(df):
    """Crée des variables métier à partir des données brutes"""
    fe = df.copy()
    
    # Utilisation du crédit
    fe['utilization'] = np.clip(fe['BILL_AMT1'] / (fe['LIMIT_BAL'] + 1e-6), 0, 5)
    
    # Ratios de remboursement
    fe['repay_ratio_recent'] = np.clip(fe['PAY_AMT1'] / (fe['BILL_AMT1'] + 1e-6), 0, 5)
    
    # Moyenne des remboursements vs factures
    pay_cols = [c for c in fe.columns if c.startswith('PAY_AMT')]
    bill_cols = [c for c in fe.columns if c.startswith('BILL_AMT')]
    fe['repay_ratio_mean'] = np.clip(
        fe[pay_cols].sum(axis=1) / (fe[bill_cols].sum(axis=1) + 1e-6), 0, 5
    )
    
    # Comptage des retards
    pay_status_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
    available_pay_cols = [c for c in pay_status_cols if c in fe.columns]
    fe['nb_retards'] = (fe[available_pay_cols] > 0).sum(axis=1)
    fe['retard_recent'] = (fe['PAY_0'] > 0).astype(int) if 'PAY_0' in fe.columns else 0
    
    # Montants moyens
    fe['facture_moyenne'] = fe[bill_cols].mean(axis=1)
    fe['paiement_moyen'] = fe[pay_cols].mean(axis=1)
    
    # Tendances (évolution récente)
    if 'BILL_AMT3' in fe.columns:
        fe['tendance_facture'] = (fe['BILL_AMT1'] - fe['BILL_AMT3']) / (np.abs(fe['BILL_AMT3']) + 1e-6)
    if 'PAY_AMT3' in fe.columns:
        fe['tendance_paiement'] = (fe['PAY_AMT1'] - fe['PAY_AMT3']) / (np.abs(fe['PAY_AMT3']) + 1e-6)
    
    return fe

# Création des features
df_enrichi = creer_features_metier(df)

# Variables métier créées
nouvelles_vars = ['utilization', 'repay_ratio_recent', 'repay_ratio_mean', 
                  'nb_retards', 'retard_recent', 'facture_moyenne', 
                  'paiement_moyen', 'tendance_facture', 'tendance_paiement']
nouvelles_vars = [v for v in nouvelles_vars if v in df_enrichi.columns]

print("🎯 Variables métier créées:")
for var in nouvelles_vars:
    print(f"  • {var}")

🎯 Variables métier créées:
  • utilization
  • repay_ratio_recent
  • repay_ratio_mean
  • nb_retards
  • retard_recent
  • facture_moyenne
  • paiement_moyen
  • tendance_facture
  • tendance_paiement


### 🎯 **Variables métier créées pour segmenter les clients de crédit**

**📊 Utilisation du crédit :**
- `utilization` : Mesure à quel point le client utilise sa limite de crédit (facture récente / limite totale)

**💰 Comportement de remboursement :**
- `repay_ratio_recent` : Capacité de remboursement immédiate (paiement récent / facture récente)
- `repay_ratio_mean` : Habitude générale de remboursement (moyenne des paiements / moyenne des factures)

**⚠️ Profil de risque :**
- `nb_retards` : Nombre total de mois où le client a été en retard
- `retard_recent` : Indicateur binaire de retard dans le mois actuel

**📈 Niveau d'activité :**
- `facture_moyenne` : Montant moyen dépensé par mois
- `paiement_moyen` : Montant moyen remboursé par mois

**📊 Tendances comportementales :**
- `tendance_facture` : Évolution récente des dépenses (augmentation/diminution)
- `tendance_paiement` : Évolution récente des remboursements

**🎯 Objectif :** Ces variables permettent de capturer 3 dimensions clés pour la segmentation :
1. **Valeur client** (montants, utilisation)
2. **Risque** (retards, ratios de remboursement) 
3. **Dynamique** (tendances d'évolution)

Ces indicateurs métier sont plus parlants que les variables brutes pour créer des segments actionnables en marketing.

## 📈 Étape 4: Analyse des Relations

Explorez les relations entre variables pour comprendre les profils clients.

In [11]:
@interact(
    var_x=Dropdown(options=nouvelles_vars, value='utilization', description='Variable X:'),
    var_y=Dropdown(options=nouvelles_vars, value='repay_ratio_recent', description='Variable Y:'),
    type_graph=Dropdown(options=['Scatter', 'Hexbin', 'Corrélation'], value='Scatter', description='Type:')
)
def analyser_relations(var_x, var_y, type_graph):
    """Analyse interactive des relations entre variables"""
    plt.figure(figsize=(10, 6))
    
    if type_graph == 'Scatter':
        plt.scatter(df_enrichi[var_x], df_enrichi[var_y], alpha=0.6, s=20)
        plt.xlabel(var_x)
        plt.ylabel(var_y)
        plt.title(f'Relation: {var_x} vs {var_y}')
        
        # Calcul corrélation
        corr = df_enrichi[[var_x, var_y]].corr().iloc[0, 1]
        plt.text(0.05, 0.95, f'Corrélation: {corr:.3f}', 
                transform=plt.gca().transAxes, bbox=dict(boxstyle="round", facecolor='wheat'))
        
    elif type_graph == 'Hexbin':
        plt.hexbin(df_enrichi[var_x], df_enrichi[var_y], gridsize=20, cmap='Blues')
        plt.colorbar(label='Nombre de points')
        plt.xlabel(var_x)
        plt.ylabel(var_y)
        plt.title(f'Densité: {var_x} vs {var_y}')
        
    elif type_graph == 'Corrélation':
        # Matrice de corrélation des variables métier
        corr_matrix = df_enrichi[nouvelles_vars].corr()
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
                   square=True, fmt='.2f')
        plt.title('Matrice de corrélation - Variables métier')
    
    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='Variable X:', options=('utilization', 'repay_ratio_recent', 'repay…

## 🎯 Étape 5: Sélection des Variables pour le Clustering

Choisissez les variables les plus pertinentes pour segmenter vos clients.

In [12]:
# Variables candidates pour le clustering
variables_base = ['AGE', 'SEX', 'EDUCATION', 'MARRIAGE', 'LIMIT_BAL']
variables_base = [v for v in variables_base if v in df_enrichi.columns]

toutes_variables = variables_base + nouvelles_vars

@interact(
    variables=SelectMultiple(
        options=toutes_variables,
        value=['AGE', 'utilization', 'repay_ratio_recent', 'nb_retards'],
        description='Variables:',
        rows=10
    )
)
def preparer_clustering(variables):
    """Prépare les données pour le clustering"""
    if len(variables) < 2:
        print("⚠️ Sélectionnez au moins 2 variables")
        return
    
    # Données pour clustering
    X_clustering = df_enrichi[list(variables)].dropna()
    
    print(f"✅ {len(variables)} variables sélectionnées:")
    for var in variables:
        print(f"  • {var}")
    
    print(f"\n📏 Données prêtes: {X_clustering.shape}")
    print(f"📊 Valeurs manquantes supprimées: {len(df_enrichi) - len(X_clustering)}")
    
    # Aperçu des statistiques
    print(f"\n📈 Statistiques des variables sélectionnées:")
    display(X_clustering.describe().round(2))
    
    # Sauvegarde pour la partie 2
    X_clustering.to_csv('donnees_clustering_partie1.csv', index=False)
    print(f"\n💾 Données sauvegardées: donnees_clustering_partie1.csv")
    
    return X_clustering

interactive(children=(SelectMultiple(description='Variables:', index=(0, 5, 6, 8), options=('AGE', 'SEX', 'EDU…

## ✅ Récapitulatif Partie 1

🎉 **Félicitations!** Vous avez terminé l'exploration des données.

**Ce que vous avez appris:**
- Charger et explorer un dataset réel
- Créer des variables métier pertinentes (utilisation, remboursement, risque)
- Analyser les relations entre variables
- Préparer les données pour le clustering

**Variables métier créées:**
- `utilization`: Taux d'utilisation du crédit (facture/limite)
- `repay_ratio_recent`: Ratio de remboursement récent
- `repay_ratio_mean`: Ratio de remboursement moyen
- `nb_retards`: Nombre de mois en retard
- `retard_recent`: Retard dans le mois actuel
- `facture_moyenne`: Montant moyen facturé
- `paiement_moyen`: Montant moyen remboursé
- `tendance_*`: Évolution récente des montants

**➡️ Prochaine étape:** Partie 2 - Application des algorithmes de clustering (K-Means, CAH, DBSCAN)